# LightGBM — Pipeline complet (split clusters + SHAP + comparaison Global vs Physique)

**Notebook** contient :
- split issu de `clustering_stratifie.ipynb` (`idx_train.npy` / `idx_test.npy` / `stratum_final`), comparaison features globales `X` vs features physiques `X_physical` sur ce même split,
- sélection de features par SHAP, ré-entraînement sur les top features, et diagnostics avancés (overfit, gains vs baseline, importance consensus).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb
import shap

ROOT = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / "data" / "processed"

RANDOM_STATE = 42
N_TOP_FEATURES = 30

## 1. Chargement des données et du split (issu de `clustering_stratifie.ipynb`)

In [ ]:
X = pd.read_parquet(DATA_PROCESSED / "X.parquet")
Y = pd.read_parquet(DATA_PROCESSED / "Y.parquet")
X_physical = pd.read_parquet(DATA_PROCESSED / "X_physical_engineered.parquet")
meta = pd.read_parquet(DATA_PROCESSED / "metadata_clean.parquet")
labels = pd.read_parquet(DATA_PROCESSED / "cluster_labels.parquet")

idx_train = np.load(DATA_PROCESSED / "idx_train.npy")
idx_test = np.load(DATA_PROCESSED / "idx_test.npy")

TARGET = 'out.electricity.total.energy_consumption..kwh'

print("X          :", X.shape)
print("X_physical :", X_physical.shape)
print("Y          :", Y.shape)
print("idx_train / idx_test :", idx_train.shape, idx_test.shape)

# X_physical est construit à partir de X.copy() (même index, même ordre de lignes) :
# les mêmes idx_train / idx_test s'appliquent donc directement aux deux jeux de features.
assert X.index.equals(X_physical.index), "X et X_physical n'ont pas le même index : les idx ne sont pas transposables tels quels."
assert X.index.equals(meta.index), "X et metadata n'ont pas le même index"

In [ ]:
# ============================================================
# Filtres (physique + surface)
# ============================================================

mask_physical = (
    (meta['in.geometry_building_type_recs'] == 'Single-Family Detached') &
    (X['in.geometry_stories'] == 1) &
    (meta['in.heating_fuel'] == 'Electricity') &
    (X['in.electric_vehicle_charger'] == 0) &
    (X['in.has_pool'] == 0) &
    (X['in.has_pv'] == 0)
)

surface_max = 250
mask_surface = X["in.geometry_floor_area"] < surface_max

mask_all = (mask_physical & mask_surface).values

print("Avant filtrage                  :", len(X))
print("Après filtre physique            :", int(mask_physical.values.sum()))
print("Après filtre physique + surface  :", int(mask_all.sum()))

# ============================================================
# Remapping idx_train / idx_test AVANT le reset_index
# ============================================================
# idx_train / idx_test sont des POSITIONS dans X.parquet non filtré (549 971 lignes,
# cf. clustering_stratifie.ipynb : idx = np.arange(len(X))). Une fois qu'on filtre X
# et qu'on fait reset_index, la position de chaque ligne survivante change : appliquer
# idx_train/idx_test tels quels via .iloc sur les données filtrées sélectionnerait de
# mauvaises lignes (ou lèverait une IndexError). On calcule donc la nouvelle position
# de chaque ligne survivante, puis on ne garde de idx_train/idx_test que les entrées
# qui survivent au filtre, remappées sur leur nouvelle position — l'appartenance
# train/test décidée dans clustering_stratifie.ipynb est préservée, on ne fait que la
# restreindre au sous-ensemble filtré.

new_pos = np.cumsum(mask_all) - 1  # nouvelle position (post-filtre) de chaque ligne encore présente

idx_train = new_pos[idx_train[mask_all[idx_train]]]
idx_test = new_pos[idx_test[mask_all[idx_test]]]

# ============================================================
# Application des filtres + reset_index (sûr maintenant que idx_train/idx_test sont remappés)
# ============================================================

X = X.loc[mask_all].reset_index(drop=True)
Y = Y.loc[mask_all].reset_index(drop=True)
X_physical = X_physical.loc[mask_all].reset_index(drop=True)
labels = labels.loc[mask_all].reset_index(drop=True)

# stratum_train doit être recalculé sur les idx_train remappés (post-filtre)
stratum_train = labels.iloc[idx_train]["stratum_final"]

print("\n===== DATA FINALE =====")
print("X          :", X.shape)
print("X_physical :", X_physical.shape)
print("Y          :", Y.shape)
print("labels     :", labels.shape)
print("idx_train / idx_test (remappés) :", idx_train.shape, idx_test.shape)

## 2. Fonctions réutilisables (split, baseline, SHAP, métriques, diagnostics)

In [ ]:
def build_splits(X_data, Y_data, idx_train, idx_test, stratum_train, test_size=0.2, random_state=RANDOM_STATE):
    """Construit train_final / val / test à partir d'un split fixe (idx_train/idx_test)
    et d'une stratification déjà calculée (stratum_train) pour le sous-split val."""

    X_train = X_data.iloc[idx_train]
    X_test = X_data.iloc[idx_test]
    Y_train = Y_data.iloc[idx_train]
    Y_test = Y_data.iloc[idx_test]

    # stratum_train vient de clustering_stratifie.ipynb, où chaque stratum a une taille
    # minimale garantie SUR LE DATASET COMPLET (549 971 lignes). Une fois restreint au
    # sous-ensemble filtré (physique + surface, potentiellement quelques milliers de lignes),
    # certains strata n'ont plus qu'1 membre ici, ce qui fait planter train_test_split
    # (stratify=...). On refusionne localement les strata trop rares dans un groupe "RARE"
    # (même logique que clustering_stratifie.ipynb, appliquée cette fois au sous-ensemble filtré).
    stratum_local = pd.Series(np.asarray(stratum_train)).reset_index(drop=True)
    rare = stratum_local.value_counts().loc[lambda c: c < 2].index
    stratum_local = stratum_local.where(~stratum_local.isin(rare), "RARE")

    try:
        X_train_final, X_val, Y_train_final, Y_val = train_test_split(
            X_train, Y_train,
            test_size=test_size,
            random_state=random_state,
            stratify=stratum_local,
        )
    except ValueError as e:
        print(f"[build_splits] Stratification impossible même après fusion des strata rares ({e}) "
              f"— split non stratifié pour ce sous-ensemble.")
        X_train_final, X_val, Y_train_final, Y_val = train_test_split(
            X_train, Y_train, test_size=test_size, random_state=random_state,
        )

    total = len(X_train_final) + len(X_val) + len(X_test)
    print(f"Train : {X_train_final.shape} ({len(X_train_final)/total:.1%})")
    print(f"Val   : {X_val.shape} ({len(X_val)/total:.1%})")
    print(f"Test  : {X_test.shape} ({len(X_test)/total:.1%})")

    return X_train_final, X_val, X_test, Y_train_final, Y_val, Y_test

In [ ]:
def dummy_baseline(Y_train_final, Y_test, target=TARGET):
    """Baseline naïve : prédiction = médiane du train pour une seule cible."""

    median_value = Y_train_final[target].median()

    y_pred = np.full(len(Y_test), median_value)

    results_dummy = {
        "RMSE": np.sqrt(mean_squared_error(Y_test[target], y_pred)),
        "MAE": mean_absolute_error(Y_test[target], y_pred),
        "R2": r2_score(Y_test[target], y_pred)
    }

    return pd.DataFrame([results_dummy], index=[target])

In [ ]:

def train_with_shap_selection(
    X_train_final, X_val, X_test,
    Y_train_final, Y_val,
    target=TARGET,
    n_top_features=N_TOP_FEATURES,
    random_state=RANDOM_STATE
):
    """
    Entraîne un LightGBM sur toutes les features,
    calcule l'importance SHAP,
    sélectionne les meilleures features,
    puis ré-entraîne un modèle final allégé.
    """

    print(f"\n========== INITIAL MODEL {target} ==========")

    # ============================
    # 1) Modèle initial
    # ============================

    model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(
        X_train_final,
        Y_train_final[target],
        eval_set=[(X_val, Y_val[target])],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    prediction = model.predict(X_test)


    # ============================
    # 2) SHAP feature importance
    # ============================

    print("\nCalcul SHAP...")

    explainer = shap.TreeExplainer(model)

    # échantillon pour accélérer le calcul SHAP
    X_shap = X_train_final.sample(
        min(20000, len(X_train_final)),
        random_state=random_state
    )

    shap_values = explainer.shap_values(X_shap)

    shap_importance = np.abs(shap_values).mean(axis=0)

    importance_df = pd.DataFrame({
        "feature": X_train_final.columns,
        "importance": shap_importance
    }).sort_values(
        "importance",
        ascending=False
    )

    top_features = importance_df["feature"].head(n_top_features).values


    # ============================
    # 3) Modèle final réduit
    # ============================

    print("\n========== FINAL MODEL ==========")

    X_tr = X_train_final[top_features]
    X_vl = X_val[top_features]
    X_te = X_test[top_features]

    final_model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        random_state=random_state,
        n_jobs=-1
    )

    final_model.fit(
        X_tr,
        Y_train_final[target],
        eval_set=[(X_vl, Y_val[target])],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    final_prediction = final_model.predict(X_te)


    return {
        "model": model,
        "prediction": prediction,
        "importance_df": importance_df,
        "top_features": top_features,
        "final_model": final_model,
        "final_prediction": final_prediction
    }

In [ ]:
def compute_metrics(Y_test, y_pred, target=TARGET):
    """
    Calcule RMSE, MAE, R2 et les erreurs relatives
    pour une seule cible.
    """

    y_true = Y_test[target]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mean_target = y_true.mean()

    results = {
        "Mean": mean_target,
        "RMSE": rmse,
        "MAE": mae,
        "RMSE (%)": 100 * rmse / mean_target,
        "MAE (%)": 100 * mae / mean_target,
        "R2": r2_score(y_true, y_pred)
    }

    return pd.DataFrame([results], index=[target])

In [ ]:
def overfit_gap(final_model,X_train_final,Y_train_final,Y_test,final_prediction,top_features,target=TARGET):
    """
    Calcule l'écart R2 train/test pour détecter le sur-apprentissage.
    """

    # prédictions train avec les features sélectionnées
    y_train_pred = final_model.predict(
        X_train_final[top_features]
    )

    # prédictions test
    y_test_pred = final_prediction

    train_r2 = r2_score(
        Y_train_final[target],
        y_train_pred
    )

    test_r2 = r2_score(
        Y_test[target],
        y_test_pred
    )

    results = {
        "Train_R2": train_r2,
        "Test_R2": test_r2,
        "Gap": train_r2 - test_r2
    }

    return pd.DataFrame([results], index=[target])

In [ ]:
def feature_importance(model, X_train_final, n_top=107):
    """
    Retourne les n features les plus importantes
    pour un modèle LightGBM unique.
    """

    importance_df = pd.DataFrame({
        "Variable": X_train_final.columns,
        "Importance": model.feature_importances_
    })

    importance_df = importance_df.sort_values(
        "Importance",
        ascending=False
    ).reset_index(drop=True)

    return importance_df.head(n_top)

In [ ]:
def gain_vs_dummy(df_dummy, df_metrics, target=TARGET):
    """
    Compare une baseline naïve (médiane) avec le modèle LightGBM final
    pour une seule cible.
    """

    comparison = pd.DataFrame(index=[target])

    comparison["Dummy_RMSE"] = df_dummy.loc[target, "RMSE"]
    comparison["Dummy_MAE"] = df_dummy.loc[target, "MAE"]
    comparison["Dummy_R2"] = df_dummy.loc[target, "R2"]

    comparison["LGBM_RMSE"] = df_metrics.loc[target, "RMSE"]
    comparison["LGBM_MAE"] = df_metrics.loc[target, "MAE"]
    comparison["LGBM_R2"] = df_metrics.loc[target, "R2"]

    comparison["Gain_RMSE_%"] = (
        (comparison["Dummy_RMSE"] - comparison["LGBM_RMSE"])
        / comparison["Dummy_RMSE"]
        * 100
    )

    comparison["Gain_MAE_%"] = (
        (comparison["Dummy_MAE"] - comparison["LGBM_MAE"])
        / comparison["Dummy_MAE"]
        * 100
    )

    comparison["Gain_R2"] = (
        comparison["LGBM_R2"] - comparison["Dummy_R2"]
    )

    return comparison

## 3. Exécution — Features globales (`X`)

In [ ]:
X_train_final_g, X_val_g, X_test_g, Y_train_final_g, Y_val_g, Y_test_g = build_splits(
    X, Y, idx_train, idx_test, stratum_train
)

In [ ]:
df_dummy_global = dummy_baseline(Y_train_final_g, Y_test_g, TARGET)
df_dummy_global

In [ ]:
out_global = train_with_shap_selection(
    X_train_final_g, X_val_g, X_test_g, Y_train_final_g, Y_val_g, TARGET
)

In [ ]:
df_metrics_global = compute_metrics(
    Y_test_g,
    out_global["final_prediction"],
    TARGET
)
df_metrics_global.round(2)

In [ ]:
df_overfit_global = overfit_gap(
    final_model=out_global["final_model"],
    X_train_final=X_train_final_g,
    Y_train_final=Y_train_final_g,
    Y_test=Y_test_g,
    final_prediction=out_global["final_prediction"],
    top_features=out_global["top_features"],
    target=TARGET
)

df_overfit_global

In [ ]:
importance_global = feature_importance(
    out_global["final_model"],
    X_train_final_g[out_global["top_features"]],
    n_top=20
)

importance_global

In [ ]:
#============================================================
# Scatter réel vs prédit — linéaire et log
# MODÈLE FINAL (post-sélection SHAP)
#============================================================

rng = np.random.default_rng(RANDOM_STATE)
n_points = 3000

target = TARGET

# Vraies valeurs et prédictions du modèle final
y_true = Y_test_g[target].reset_index(drop=True)
y_pred = pd.Series(out_global["final_prediction"]).reset_index(drop=True)


# ==========================
# Échantillonnage
# ==========================

n = min(n_points, len(y_true))

sample = rng.choice(len(y_true),size=n,replace=False)

y_true_s = y_true.iloc[sample]
y_pred_s = y_pred.iloc[sample]


# ==========================
# Suppression valeurs <= 0
# (nécessaire pour l'échelle log)
# ==========================

mask = (y_true_s > 0) & (y_pred_s > 0)

y_true_s = y_true_s[mask]
y_pred_s = y_pred_s[mask]


min_val = min(y_true_s.min(),y_pred_s.min())

max_val = max(y_true_s.max(),y_pred_s.max())


# ==========================
# Graphiques
# ==========================

fig, axes = plt.subplots(1,2,figsize=(12, 5))


# ---- Linéaire ----

axes[0].scatter(y_true_s,y_pred_s,alpha=0.3,s=10)
axes[0].plot([min_val, max_val],[min_val, max_val],"r--",linewidth=2)
axes[0].set_title("Échelle linéaire")
axes[0].set_xlabel("Réel (kWh/an)")
axes[0].set_ylabel("Prédit (kWh/an)")
axes[0].grid(alpha=0.3)



# ---- Log ----

axes[1].scatter(y_true_s,y_pred_s,alpha=0.3,s=10)
axes[1].plot([min_val, max_val],[min_val, max_val],"r--",linewidth=2)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Échelle log")
axes[1].set_xlabel("Réel (kWh/an)")
axes[1].set_ylabel("Prédit (kWh/an)")
axes[1].grid( alpha=0.3,which="both")


plt.suptitle(f"Modèle final LightGBM + sélection SHAP\n{target}")
plt.tight_layout()
plt.savefig(f"scatter_real_pred_{target}.png",dpi=300,bbox_inches="tight")
plt.show()

## 4. Exécution — Features physiques (`X_physical`)

Même split (`idx_train` / `idx_test` / `stratum_final`), mêmes fonctions, appliqués à `X_physical` pour une comparaison strictement équivalente.

In [ ]:
X_train_final_p, X_val_p, X_test_p, Y_train_final_p, Y_val_p, Y_test_p = build_splits(
    X_physical, Y, idx_train, idx_test, stratum_train
)

In [ ]:
df_dummy_physical = dummy_baseline(Y_train_final_p, Y_test_p, TARGET)
df_dummy_physical

In [ ]:
out_physical = train_with_shap_selection(
    X_train_final_p, X_val_p, X_test_p, Y_train_final_p, Y_val_p, TARGET
)

In [ ]:

df_metrics_physical = compute_metrics(
    Y_test_g,
    out_physical["final_prediction"],
    TARGET
)
df_metrics_physical.round(2)

In [ ]:


df_overfit_physical = overfit_gap(
    final_model=out_physical["final_model"],
    X_train_final=X_train_final_p,
    Y_train_final=Y_train_final_p,
    Y_test=Y_test_p,
    final_prediction=out_physical["final_prediction"],
    top_features=out_physical["top_features"],
    target=TARGET
)

df_overfit_physical

In [ ]:
importance_physical = feature_importance(
    out_physical["final_model"],
    X_train_final_p[out_physical["top_features"]],
    n_top=82
)

importance_physical

## 5. Comparaison finale — Global vs Physique (même split, même méthodologie)

In [ ]:
final_comparison = pd.DataFrame({
    "R2_global": df_metrics_global["R2"],
    "R2_physical": df_metrics_physical["R2"],
    "RMSE_global": df_metrics_global["RMSE"],
    "RMSE_physical": df_metrics_physical["RMSE"],
    "Overfit_gap_global": df_overfit_global["Gap"],
    "Overfit_gap_physical": df_overfit_physical["Gap"],
})

final_comparison["delta_R2 (physical - global)"] = (
    final_comparison["R2_physical"] - final_comparison["R2_global"]
)
final_comparison["delta_RMSE (physical - global)"] = (
    final_comparison["RMSE_physical"] - final_comparison["RMSE_global"]
)

final_comparison.sort_values("delta_R2 (physical - global)", ascending=True).round(4)

## 6. Pistes d'optimisation restantes

1. **Optuna** n'est utilisé nulle part ici non plus : les hyperparamètres (`n_estimators=1000`, `learning_rate=0.05`, `num_leaves=63`) sont fixes et identiques pour les 5 cibles. Une recherche Optuna par cible (surtout `fuel_oil` et `propane`, qui ont le plus gros écart train/test) reste la prochaine étape logique.
2. **`fuel_oil` et `propane`** ont systématiquement le R² le plus bas et le `Gap` train/test le plus élevé (cf. section overfit) → cibles prioritaires pour un objectif Tweedie/Poisson ou une régularisation plus forte, plutôt que la MSE par défaut.
3. **`comparison_global` / `comparison_physical` / `final_comparison`** te permettent de trancher objectivement si `X_physical` apporte quelque chose sur ce split — regarde en particulier `delta_R2` pour chaque cible.